# AD9226 Capture Full-Chain Test

Use IP-local offsets with `ip.write(offset, value)`. Do not add the Vivado base address.

Recommended verification order:
1. `capture_mode = 0`: HLS writer fake -> DDR.
2. `capture_mode = 2`: capture_core fake stream -> FIFO -> HLS stream -> DDR.
3. `capture_mode = 1`: real AD9226 -> FIFO -> HLS stream -> DDR.

In [ ]:
from time import time
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

overlay = Overlay('base_add.bit')
print('IP names:', list(overlay.ip_dict.keys()))
ctrl = overlay.adc_capture_0
writer = overlay.base_add_0
print('adc_capture_0 base:', hex(overlay.ip_dict['adc_capture_0']['phys_addr']))
print('base_add_0 base:', hex(overlay.ip_dict['base_add_0']['phys_addr']))
print('Use IP-local offsets only.')

In [ ]:
MAX_SAMPLE_N = 65536
BUFFER_WORDS = 2 * MAX_SAMPLE_N

CTRL = 0x00
STATUS = 0x04
SAMPLE_COUNT = 0x08
ADC_HALF = 0x0C
SAMPLE_DELAY = 0x10
DECIMATION = 0x14
CHANNEL_MASK = 0x18
CAPTURE_MODE = 0x1C
TRIGGER_MODE = 0x20
PRE_DELAY = 0x24
BUFFER_SELECT = 0x28
LATEST_A = 0x2C
LATEST_B = 0x30
SAMPLE_COUNTER = 0x34
FIFO_LEVEL = 0x38
ERROR_FLAGS = 0x3C
LED_CTRL = 0x40
VERSION = 0x44
SAVED_COUNTER = 0x48
LAST_SAMPLE_WORD = 0x4C
DEBUG_STATE = 0x50

WRITER_AP_CTRL = 0x00
WRITER_BUFFER = 0x10
WRITER_COUNT = 0x18
WRITER_CAPTURE_MODE = 0x20

def dump_regs():
    status = ctrl.read(STATUS)
    error_flags = ctrl.read(ERROR_FLAGS)
    fifo_level = ctrl.read(FIFO_LEVEL)
    saved_counter = ctrl.read(SAVED_COUNTER)
    last_sample = ctrl.read(LAST_SAMPLE_WORD)
    debug_state = ctrl.read(DEBUG_STATE)
    writer_ctrl = writer.read(WRITER_AP_CTRL)
    print('STATUS          = 0x%08X' % status)
    print('busy/done/clk   =', status & 1, (status >> 1) & 1, (status >> 2) & 1)
    print('ERROR_FLAGS     = 0x%08X' % error_flags)
    print('FIFO_LEVEL      =', fifo_level)
    print('SAVED_COUNTER   =', saved_counter)
    print('LAST_SAMPLE     = 0x%08X' % last_sample)
    print('DEBUG_STATE     =', debug_state)
    print('WRITER_AP_CTRL  = 0x%08X' % writer_ctrl)
    print('LATEST_A        =', ctrl.read(LATEST_A))
    print('LATEST_B        =', ctrl.read(LATEST_B))
    print('SAMPLE_COUNTER  =', ctrl.read(SAMPLE_COUNTER))
    print('VERSION         = 0x%08X' % ctrl.read(VERSION))

dump_regs()

In [ ]:
sample_count = 1024  # quick test; real tests can use 16384 / 32768 / 65536
sample_count = min(max(int(sample_count), 1), MAX_SAMPLE_N)

target_adc_fs = 10_000_000
adc_half_period = round(125_000_000 / (2 * target_adc_fs))
adc_half_period = max(1, int(adc_half_period))
actual_adc_fs = 125_000_000 / (2 * adc_half_period)

sample_delay = 1
decimation = 1
channel_mask = 0b11
trigger_mode = 0
pre_delay = 0
buffer_select = 0
led_ps_override = 0
led_value = 0

print('target_adc_fs =', target_adc_fs)
print('adc_half_period =', adc_half_period)
print('actual_adc_fs = %.3f MHz' % (actual_adc_fs / 1e6))
print('sample_count =', sample_count)

In [ ]:
buf = allocate(shape=(BUFFER_WORDS,), dtype=np.int32)

def run_capture(capture_mode, timeout_s=5.0):
    global actual_adc_fs
    sample_n = min(max(int(sample_count), 1), MAX_SAMPLE_N)
    decim = max(1, int(decimation))
    half = max(1, int(adc_half_period))
    actual_adc_fs = 125_000_000 / (2 * half)

    buf[:] = -12345
    buf.flush()

    # 1. ctrl clear
    ctrl.write(CTRL, 0x04)
    ctrl.write(CTRL, 0x00)

    # 2. write ctrl parameters
    ctrl.write(SAMPLE_COUNT, sample_n)
    ctrl.write(ADC_HALF, half)
    ctrl.write(SAMPLE_DELAY, int(sample_delay))
    ctrl.write(DECIMATION, decim)
    ctrl.write(CHANNEL_MASK, int(channel_mask))
    ctrl.write(CAPTURE_MODE, int(capture_mode))
    ctrl.write(TRIGGER_MODE, int(trigger_mode))
    ctrl.write(PRE_DELAY, int(pre_delay))
    ctrl.write(BUFFER_SELECT, int(buffer_select))
    ctrl.write(LED_CTRL, (int(led_ps_override) << 8) | int(led_value))

    # 3. write writer buffer/sample_count/capture_mode
    writer.write(WRITER_BUFFER, buf.physical_address)
    writer.write(WRITER_COUNT, sample_n)
    writer.write(WRITER_CAPTURE_MODE, int(capture_mode))

    if capture_mode == 0:
        writer.write(WRITER_AP_CTRL, 0x01)
    else:
        # 4. ctrl enable, but do not start
        ctrl.write(CTRL, 0x01)
        # 5. writer ap_start
        writer.write(WRITER_AP_CTRL, 0x01)
        # 6. ctrl start pulse
        ctrl.write(CTRL, 0x03)
        ctrl.write(CTRL, 0x01)

    t0 = time()
    while True:
        ctrl_status = ctrl.read(STATUS)
        writer_ctrl = writer.read(WRITER_AP_CTRL)
        writer_done = ((writer_ctrl >> 1) & 1) != 0
        ctrl_done = ((ctrl_status >> 1) & 1) != 0
        done = writer_done if capture_mode == 0 else (writer_done and ctrl_done)
        if done:
            break
        if time() - t0 > timeout_s:
            dump_regs()
            raise TimeoutError(
                'capture timeout: STATUS=0x%08X ERROR_FLAGS=0x%08X FIFO_LEVEL=%d SAVED_COUNTER=%d LAST_SAMPLE_WORD=0x%08X DEBUG_STATE=%d WRITER_AP_CTRL=0x%08X'
                % (
                    ctrl_status,
                    ctrl.read(ERROR_FLAGS),
                    ctrl.read(FIFO_LEVEL),
                    ctrl.read(SAVED_COUNTER),
                    ctrl.read(LAST_SAMPLE_WORD),
                    ctrl.read(DEBUG_STATE),
                    writer_ctrl,
                )
            )

    buf.invalidate()
    dump_regs()
    ch0 = np.array(buf[0:sample_n], dtype=np.int32)
    ch1 = np.array(buf[MAX_SAMPLE_N:MAX_SAMPLE_N + sample_n], dtype=np.int32)
    return ch0, ch1

def plot_capture(ch0, ch1, title):
    nplot = min(len(ch0), 512)
    t = np.arange(nplot) / (actual_adc_fs / max(1, int(decimation)))
    print('CH0 first 16:', ch0[:16].tolist())
    print('CH1 first 16:', ch1[:16].tolist())
    print('CH0 mean/Vpp/RMS:', float(ch0.mean()), int(ch0.max() - ch0.min()), float(np.sqrt(np.mean((ch0 - ch0.mean()) ** 2))))
    print('CH1 mean/Vpp/RMS:', float(ch1.mean()), int(ch1.max() - ch1.min()), float(np.sqrt(np.mean((ch1 - ch1.mean()) ** 2))))
    plt.figure(figsize=(10, 4))
    plt.plot(t, ch0[:nplot], label='CH0 raw')
    plt.plot(t, ch1[:nplot], label='CH1 raw')
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
# Step 1: capture_mode = 0, HLS fake -> DDR.
ch0, ch1 = run_capture(capture_mode=0)
plot_capture(ch0, ch1, 'capture_mode=0: HLS fake -> DDR')

In [ ]:
# Step 2: capture_mode = 2, capture_core fake stream -> FIFO -> HLS stream -> DDR.
ch0, ch1 = run_capture(capture_mode=2)
plot_capture(ch0, ch1, 'capture_mode=2: capture_core fake stream -> FIFO -> HLS -> DDR')

In [ ]:
# Step 3: capture_mode = 1, real AD9226 -> FIFO -> HLS stream -> DDR.
# Connect and bias the AD9226 input correctly before running this cell.
ch0, ch1 = run_capture(capture_mode=1)
plot_capture(ch0, ch1, 'capture_mode=1: real AD9226 -> FIFO -> HLS -> DDR')